In [2]:
from pathlib import Path
import pandas as pd

def find_project_root():
    path = Path.cwd().resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError("Could not find data/raw from " + str(path))

RAW_DIR = find_project_root() / "data" / "raw"
print("Reading data from:", RAW_DIR)
FILES = {
    "2017-2020": {"demo": "P_DEMO", "bpq": "P_BPQ", "bpxo": "P_BPXO"},
    "2021-2023": {"demo": "DEMO_L", "bpq": "BPQ_L", "bpxo": "BPXO_L"},
}

def read_xpt(period, name):
    return pd.read_sas(RAW_DIR / period / f"{name}.xpt", format="xport").round(6)

def before_bp_step(period):
    """Adults >=20, not pregnant, BPQ020 == 2 (D-003), before requiring a BP reading."""
    f = FILES[period]
    demo = read_xpt(period, f["demo"])[["SEQN", "RIDAGEYR", "RIAGENDR", "RIDEXPRG", "RIDSTATR"]]
    bpq = read_xpt(period, f["bpq"])[["SEQN", "BPQ020"]]
    bpxo = read_xpt(period, f["bpxo"])[["SEQN", "BPXOSY2", "BPXODI2"]]
    df = demo.merge(bpq, on="SEQN", how="left").merge(bpxo, on="SEQN", how="left")
    df = df[(df["RIDAGEYR"] >= 20) & (df["RIDEXPRG"] != 1) & (df["BPQ020"] == 2)].copy()
    df["has_bp"] = (df["BPXOSY2"] > 0) & (df["BPXODI2"] > 0)
    df["examined"] = df["RIDSTATR"] == 2
    df["female"] = df["RIAGENDR"] == 2
    return df

for period in FILES:
    df = before_bp_step(period)
    print(f"\n=== {period} (n = {len(df)}) ===")

    print("\nWhy is BP missing?")
    print(pd.crosstab(df["examined"], df["has_bp"],
                      rownames=["Examined at MEC"], colnames=["Has BP"], margins=True))

    print("\nWith vs without BP:")
    summary = df.groupby("has_bp").agg(
        n=("SEQN", "size"),
        mean_age=("RIDAGEYR", "mean"),
        pct_age_60_plus=("RIDAGEYR", lambda a: (a >= 60).mean() * 100),
        pct_female=("female", lambda s: s.mean() * 100),
    ).round(1)
    summary.index = summary.index.map({True: "Has BP", False: "No BP"})
    print(summary)

Reading data from: C:\Users\gamal\Documents\Hypertension-Screening\data\raw

=== 2017-2020 (n = 5578) ===

Why is BP missing?
Has BP           False  True   All
Examined at MEC                   
False              427     0   427
True               554  4597  5151
All                981  4597  5578

With vs without BP:
           n  mean_age  pct_age_60_plus  pct_female
has_bp                                             
No BP    981      45.8             23.6        55.5
Has BP  4597      45.3             23.6        50.3

=== 2021-2023 (n = 4815) ===

Why is BP missing?
Has BP           False  True   All
Examined at MEC                   
False             1105     0  1105
True               123  3587  3710
All               1228  3587  4815

With vs without BP:
           n  mean_age  pct_age_60_plus  pct_female
has_bp                                             
No BP   1228      46.4             26.7        56.1
Has BP  3587      48.8             33.5        55.5


In [3]:
df = before_bp_step("2017-2020")
examined_no_bp = df[df["examined"] & ~df["has_bp"]]

# 2017-2018 participants have SEQN up to 102956; later ones are 2019-March 2020
first_cycle = examined_no_bp["SEQN"] <= 102956
print("Examined but no BP:", len(examined_no_bp))
print("  from 2017-2018:", first_cycle.sum())
print("  from 2019-2020:", (~first_cycle).sum())

Examined but no BP: 554
  from 2017-2018: 0
  from 2019-2020: 554


In [4]:
bpxo = read_xpt("2017-2020", "P_BPXO")
ids = examined_no_bp["SEQN"]
in_file = ids.isin(bpxo["SEQN"])

print("In P_BPXO file but no reading 2:", in_file.sum())
print("Not in P_BPXO file at all:      ", (~in_file).sum())

print("\nAge / sex of these 554 vs everyone with BP:")
print(examined_no_bp[["RIDAGEYR", "female"]].mean().round(2))
print(df[df["has_bp"]][["RIDAGEYR", "female"]].mean().round(2))


In P_BPXO file but no reading 2: 554
Not in P_BPXO file at all:       0

Age / sex of these 554 vs everyone with BP:
RIDAGEYR    45.47
female       0.60
dtype: float64
RIDAGEYR    45.32
female       0.50
dtype: float64


In [5]:
missing = bpxo[bpxo["SEQN"].isin(ids)]
complete = bpxo[bpxo["SEQN"].isin(df.loc[df["has_bp"], "SEQN"])]

print("How many readings do the 554 have?")
for col in ["BPXOSY1", "BPXOSY2", "BPXOSY3"]:
    print(f"  {col}: {(missing[col] > 0).sum()}")

print("\nReading 1 (systolic) — the 554 vs complete participants:")
print(pd.DataFrame({
    "missing_r2": missing["BPXOSY1"].where(missing["BPXOSY1"] > 0).describe(),
    "complete": complete["BPXOSY1"].where(complete["BPXOSY1"] > 0).describe(),
}).round(1))

print("\nArm used (BPAOARM) and cuff size (BPAOCSZ) among the 554:")
print(missing["BPAOARM"].value_counts(dropna=False))
print(missing["BPAOCSZ"].value_counts(dropna=False))

How many readings do the 554 have?
  BPXOSY1: 6
  BPXOSY2: 0
  BPXOSY3: 0

Reading 1 (systolic) — the 554 vs complete participants:
       missing_r2  complete
count         6.0    4597.0
mean        136.8     119.6
std          25.4      16.8
min         110.0      76.0
25%         115.8     108.0
50%         136.0     117.0
75%         149.5     129.0
max         176.0     220.0

Arm used (BPAOARM) and cuff size (BPAOCSZ) among the 554:
BPAOARM
b''     538
b'R'     16
Name: count, dtype: int64
BPAOCSZ
NaN    542
4.0      6
3.0      4
5.0      2
Name: count, dtype: int64
